# HR Analytics AI System
## Notebook 1: Data Ingestion with Apache Spark

In this notebook, we will:

1. Initialize a Spark Session
2. Load the IBM HR Analytics Dataset
3. Explore the basic structure of the data
4. Save the data in Parquet format for better performance

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print(f"PySpark Version: {pyspark.__version__}")

PySpark Version: 3.5.8


### 1. Initialize Spark Session

We begin by creating a Spark Session, which is the entry point for any Spark application.
The Spark Session allows us to read data, run SQL queries, and apply transformations
across distributed data using the Spark engine.

In [2]:
spark = SparkSession.builder \
    .appName("HR Analytics - Data Ingestion") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(" Spark Session Created Successfully!")
print(f"Spark Version: {spark.version}")

 Spark Session Created Successfully!
Spark Version: 3.5.8


### 2. Load Dataset

We load the IBM HR Analytics Dataset from the local data directory using Spark's CSV reader.
The header parameter ensures that the first row is treated as column names,
and inferSchema automatically detects the correct data type for each column.

In [ ]:
df = spark.read.csv(
    r"C:\HR-Analytics-AI-System\data\WA_Fn-UseC_-HR-Employee-Attrition.csv",
    header=True,
    inferSchema=True
)

print(" Dataset Loaded Successfully!")
print(f" Total Rows: {df.count()}")
print(f" Total Columns: {len(df.columns)}")

✅ Dataset Loaded Successfully!
📊 Total Rows: 1470
📋 Total Columns: 35


### 3. Explore Data

We begin exploring the dataset by displaying the first 5 rows.
This gives us an initial understanding of the data structure,
the values in each column, and how the dataset is organized.

In [ ]:
print(" First 5 Rows:")
df.show(5, truncate=False)

🔍 First 5 Rows:
+---+---------+-----------------+---------+----------------------+----------------+---------+--------------+-------------+--------------+-----------------------+------+----------+--------------+--------+---------------------+---------------+-------------+-------------+-----------+------------------+------+--------+-----------------+-----------------+------------------------+-------------+----------------+-----------------+---------------------+---------------+--------------+------------------+-----------------------+--------------------+
|Age|Attrition|BusinessTravel   |DailyRate|Department            |DistanceFromHome|Education|EducationField|EmployeeCount|EmployeeNumber|EnvironmentSatisfaction|Gender|HourlyRate|JobInvolvement|JobLevel|JobRole              |JobSatisfaction|MaritalStatus|MonthlyIncome|MonthlyRate|NumCompaniesWorked|Over18|OverTime|PercentSalaryHike|PerformanceRating|RelationshipSatisfaction|StandardHours|StockOptionLevel|TotalWorkingYears|TrainingTimesL

### 4. Column Overview

Before diving into the analysis, it is essential to understand the structure of the dataset.
The dataset contains 35 columns that can be grouped into the following categories:

- **Employee Demographics:** Age, Gender, MaritalStatus, Education, EducationField
- **Job Information:** Department, JobRole, JobLevel, BusinessTravel, OverTime
- **Compensation:** MonthlyIncome, DailyRate, HourlyRate, MonthlyRate, PercentSalaryHike
- **Satisfaction and Performance:** PerformanceRating, JobSatisfaction, EnvironmentSatisfaction, JobInvolvement
- **Experience:** TotalWorkingYears, YearsAtCompany, YearsInCurrentRole, NumCompaniesWorked
- **Target Variable:** Attrition — whether the employee left the company (Yes) or stayed (No)

In [5]:
print("Total Number of Columns:", len(df.columns))
print("-" * 40)
for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}")

Total Number of Columns: 35
----------------------------------------
 1. Age
 2. Attrition
 3. BusinessTravel
 4. DailyRate
 5. Department
 6. DistanceFromHome
 7. Education
 8. EducationField
 9. EmployeeCount
10. EmployeeNumber
11. EnvironmentSatisfaction
12. Gender
13. HourlyRate
14. JobInvolvement
15. JobLevel
16. JobRole
17. JobSatisfaction
18. MaritalStatus
19. MonthlyIncome
20. MonthlyRate
21. NumCompaniesWorked
22. Over18
23. OverTime
24. PercentSalaryHike
25. PerformanceRating
26. RelationshipSatisfaction
27. StandardHours
28. StockOptionLevel
29. TotalWorkingYears
30. TrainingTimesLastYear
31. WorkLifeBalance
32. YearsAtCompany
33. YearsInCurrentRole
34. YearsSinceLastPromotion
35. YearsWithCurrManager


### 5. Dataset Schema

The schema provides a detailed view of the data type assigned to each column.
This step is crucial as it helps us identify which columns are numerical
and which are categorical, which directly impacts our preprocessing
and model building strategy in the upcoming notebooks.

In [6]:
print("Dataset Schema:")
print("-" * 40)
df.printSchema()

Dataset Schema:
----------------------------------------
root
 |-- Age: integer (nullable = true)
 |-- Attrition: string (nullable = true)
 |-- BusinessTravel: string (nullable = true)
 |-- DailyRate: integer (nullable = true)
 |-- Department: string (nullable = true)
 |-- DistanceFromHome: integer (nullable = true)
 |-- Education: integer (nullable = true)
 |-- EducationField: string (nullable = true)
 |-- EmployeeCount: integer (nullable = true)
 |-- EmployeeNumber: integer (nullable = true)
 |-- EnvironmentSatisfaction: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- HourlyRate: integer (nullable = true)
 |-- JobInvolvement: integer (nullable = true)
 |-- JobLevel: integer (nullable = true)
 |-- JobRole: string (nullable = true)
 |-- JobSatisfaction: integer (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- MonthlyIncome: integer (nullable = true)
 |-- MonthlyRate: integer (nullable = true)
 |-- NumCompaniesWorked: integer (nullable = true)
 |-

### 6. Basic Statistics

Descriptive statistics provide a quick summary of the numerical columns,
including the count, mean, standard deviation, minimum, and maximum values.
This helps us understand the distribution of each feature
and detect any potential outliers or anomalies in the data.

In [7]:
print("Basic Statistics:")
print("-" * 40)
df.describe().show()

Basic Statistics:
----------------------------------------
+-------+------------------+---------+--------------+------------------+---------------+----------------+------------------+----------------+-------------+-----------------+-----------------------+------+------------------+------------------+------------------+--------------------+------------------+-------------+-----------------+------------------+------------------+------+--------+------------------+-------------------+------------------------+-------------+------------------+------------------+---------------------+------------------+------------------+------------------+-----------------------+--------------------+
|summary|               Age|Attrition|BusinessTravel|         DailyRate|     Department|DistanceFromHome|         Education|  EducationField|EmployeeCount|   EmployeeNumber|EnvironmentSatisfaction|Gender|        HourlyRate|    JobInvolvement|          JobLevel|             JobRole|   JobSatisfaction|MaritalStatu

### 7. Save Data as Parquet

We save the dataset in Parquet format for the following reasons:

- Parquet is a columnar storage format optimized for big data processing
- It is significantly faster to read and write compared to CSV
- It preserves the schema and data types automatically
- It reduces storage size through built-in compression
- It is the recommended format when working with Apache Spark

In [8]:
df.write.mode("overwrite").parquet(
    r"C:\HR-Analytics-AI-System\data\hr_data.parquet"
)

print("Data saved successfully as Parquet format.")
print(f"Location: C:\\HR-Analytics-AI-System\\data\\hr_data.parquet")

Data saved successfully as Parquet format.
Location: C:\HR-Analytics-AI-System\data\hr_data.parquet


### 8. Summary

In this notebook, we successfully completed the data ingestion phase:

1. Initialized an Apache Spark Session
2. Loaded the IBM HR Analytics Dataset containing 1,470 employees and 35 columns
3. Explored the basic structure, schema, and statistics of the dataset
4. Identified numerical and categorical columns
5. Saved the dataset in Parquet format for optimized performance in subsequent steps

The dataset is now ready for the next phase: Data Preprocessing.